# Actividad 09: Análisis del Dataset del Titanic
**Curso:** Agentes Inteligentes  
**Alumno:** Quispe Bartolo, Carlos Martin

---
## Paso 1 — Plantear el problema
* **¿Qué pregunta quieres responder?** ¿Cuáles fueron las características y factores socio-demográficos (como el género, la edad o el nivel socioeconómico) que determinaron de manera estadística la probabilidad de supervivencia de un pasajero a bordo del Titanic?
* **¿Cuál es la variable objetivo?** La variable objetivo es `Survived` (0 = No sobrevivió, 1 = Sobrevivió).
* **Hipótesis iniciales:** 
    1. *Hipótesis de evacuación:* El protocolo "mujeres y niños primero" generará una tasa de supervivencia significativamente superior en el sexo femenino y en los niños.
    2. *Hipótesis socioeconómica:* Los pasajeros que viajaban en primera clase (`Pclass` = 1) tendrán una probabilidad de supervivencia mayor debido a la cercanía de sus camarotes a la cubierta.

In [3]:
# Paso 2
# 2.1. Cargar
import pandas as pd

# Cargar el dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Inspección inicial
display(df.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# 2.2. Explorar
# Información estructural y conteo de nulos
print("--- Información de Columnas ---")
df.info()

print("\n--- Estadísticas Descriptivas ---")
print(df.describe())

print("\n--- Categoría Sexo ---")
print(df['Sex'].value_counts())

print("\n--- Categoría Embarque ---")
print(df['Embarked'].value_counts())

--- Información de Columnas ---
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

--- Estadísticas Descriptivas ---
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.6991

In [5]:
# Paso 3 — Limpiar

# Imputar Age con la mediana
df['Age'] = df['Age'].fillna(df['Age'].median())

# Imputar Embarked con la moda
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Eliminar Cabin (tiene > 77% de nulos)
df = df.drop(columns='Cabin')

# Verificar nulos y duplicados
print("Valores nulos restantes:\n", df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())

Valores nulos restantes:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Filas duplicadas: 0


In [6]:
# Paso 4 — Transformar (Feature Engineering)

# Tamaño de Familia
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Es Solitario (IsAlone)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Grupos de Edad
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 60, 100], labels=['Niño', 'Adolescente', 'Adulto', 'Mayor'])

# Extraer Títulos
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
print("--- Frecuencia de Títulos ---")
print(df['Title'].value_counts().head(10))

--- Frecuencia de Títulos ---
Title
Mr        517
Miss      182
Mrs       125
Master     40
Dr          7
Rev         6
Major       2
Mlle        2
Col         2
Don         1
Name: count, dtype: int64


In [7]:
# Paso 5 — Analizar con groupby

print("=== SUPERVIVENCIA POR GÉNERO ===")
print(df.groupby('Sex')['Survived'].mean())

print("\n=== SUPERVIVENCIA POR CLASE ===")
print(df.groupby('Pclass')['Survived'].mean())

print("\n=== SUPERVIVENCIA POR GRUPO DE EDAD ===")
print(df.groupby('AgeGroup')['Survived'].mean())

print("\n=== SUPERVIVENCIA COMBINADA (GÉNERO Y CLASE) ===")
display(df.groupby(['Sex', 'Pclass'])['Survived'].agg(['mean', 'count']))

=== SUPERVIVENCIA POR GÉNERO ===
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

=== SUPERVIVENCIA POR CLASE ===
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

=== SUPERVIVENCIA POR GRUPO DE EDAD ===
AgeGroup
Niño           0.579710
Adolescente    0.428571
Adulto         0.365753
Mayor          0.227273
Name: Survived, dtype: float64

=== SUPERVIVENCIA COMBINADA (GÉNERO Y CLASE) ===


mean  count
Sex    Pclass                 
female 1       0.968085     94
       2       0.921053     76
       3       0.500000    144
male   1       0.368852    122
       2       0.157407    108
       3       0.135447    347

## Paso 6 — Conclusiones

1. **Los 3 factores con mayor impacto:** El **Género** fue el factor absoluto (las mujeres sobrevivieron en un 74%, frente al 18% de los hombres). La **Clase** del boleto también fue determinante (la 1ra clase tuvo un 62% de supervivencia frente al 24% de la 3ra). Finalmente, el **Grupo de Edad** benefició claramente a los niños (58% de supervivencia).
2. **Validación de Hipótesis:** Las hipótesis iniciales se cumplieron. Los datos respaldan estadísticamente que la regla marítima de priorizar mujeres y niños fue aplicada, y se evidencia una marcada desigualdad donde el poder adquisitivo (Primera Clase) duplicaba las probabilidades de sobrevivir.
3. **Impacto de la Limpieza:** Rellenar la columna `Age` utilizando la mediana (28 años) inyectó casi un 20% de datos artificiales en la categoría "Adulto". Esto pudo haber suavizado ligeramente la tasa de supervivencia real de ese grupo, ya que asumimos la edad de pasajeros de los cuales no teníamos certeza absoluta.

In [8]:
# Paso 7 — Exportar
df.to_csv("titanic_clean.csv", index=False)
print("Archivo exportado exitosamente.")

Archivo exportado exitosamente.
